In [1]:
import pandas as pd

# Load the leads.csv file into a DataFrame
df_leads = pd.read_csv('/content/leads.csv')

# Display the first 5 rows of the DataFrame
display(df_leads.head())

,lead_id,created_at,source,city,area,property_type,budget_pkr_lac,bedrooms,first_response_minutes,calls_made,total_call_seconds,whatsapp_replies,site_visits,agent_experience_years,is_overseas,referred_by_existing_client,has_financing_approved,token_amount_received_pkr,crm_record_hash,converted
0,MGC-104067,2025-09-30 19:35:11,Facebook Ads,Rawalpindi,Bahria Town,Farmhouse,801.0,1.0,12.0,2,72.0,4,1,4.7,0,0,1,0.0,8637176927,0
1,MGC-108870,2024-05-25 14:40:03,Instagram,Islamabad,Bahria Town,Commercial Shop,140.0,NaN,97.0,1,57.0,0,1,2.0,0,0,0,0.0,5937285284,0
2,MGC-101529,2024-07-30 02:06:11,Facebook Ads,Faisalabad,DHA,Penthouse,325.0,1.0,38.0,0,0.0,2,0,0.2,1,0,1,0.0,9807118704,0
3,MGC-108073,2025-05-08 22:01:20,Google Search,Islamabad,B-17,Apartment,85.0,3.0,15.0,1,138.0,3,0,NaN,1,0,1,0.0,6076182026,0
4,MGC-107878,2025-01-26 23:14:14,Instagram,Islamabad,Top City,Plot,117.0,NaN,27.0,2,54.0,2,0,6.3,0,0,0,0.0,4881826037,0


In [2]:
# Display basic information about the DataFrame, including data types and non-null values
df_leads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9160 entries, 0 to 9159
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   lead_id                      9160 non-null   object 
 1   created_at                   9160 non-null   object 
 2   source                       9160 non-null   object 
 3   city                         9160 non-null   object 
 4   area                         8683 non-null   object 
 5   property_type                9160 non-null   object 
 6   budget_pkr_lac               8876 non-null   float64
 7   bedrooms                     5558 non-null   float64
 8   first_response_minutes       8984 non-null   float64
 9   calls_made                   9160 non-null   int64  
 10  total_call_seconds           9160 non-null   float64
 11  whatsapp_replies             9160 non-null   int64  
 12  site_visits                  9160 non-null   int64  
 13  agent_experience_y

## Minimal SQL Schema (schema.sql)

Here is a minimal SQL schema for the `leads` data. A single table is sufficient, with `lead_id` as the primary key to prevent duplicate entries.

```sql
CREATE TABLE leads (
    lead_id VARCHAR(255) PRIMARY KEY, -- Unique identifier for each lead. This column prevents duplicate leads.
    created_at TIMESTAMP WITH TIME ZONE,
    source VARCHAR(255),
    city VARCHAR(255),
    area VARCHAR(255),
    property_type VARCHAR(255),
    budget_pkr_lac DOUBLE PRECISION,
    bedrooms DOUBLE PRECISION,
    first_response_minutes DOUBLE PRECISION,
    calls_made INTEGER,
    total_call_seconds DOUBLE PRECISION,
    whatsapp_replies INTEGER,
    site_visits INTEGER,
    agent_experience_years DOUBLE PRECISION,
    is_overseas INTEGER,
    referred_by_existing_client INTEGER,
    has_financing_approved INTEGER,
    token_amount_received_pkr DOUBLE PRECISION,
    crm_record_hash VARCHAR(255), -- Could also be made UNIQUE if it truly represents a unique CRM record hash
    converted INTEGER
);
```

## SQL Queries (queries.sql)

Here are the two requested SQL queries, ready to be executed against a database using the schema defined above.

### 1. Conversion Rate by Lead Source (200+ leads only, best first)

```sql
SELECT
    source,
    COUNT(lead_id) AS total_leads,
    SUM(CASE WHEN converted = 1 THEN 1 ELSE 0 END) AS converted_leads,
    (CAST(SUM(CASE WHEN converted = 1 THEN 1 ELSE 0 END) AS NUMERIC) * 100.0 / COUNT(lead_id)) AS conversion_rate
FROM
    leads
GROUP BY
    source
HAVING
    COUNT(lead_id) >= 200
ORDER BY
    conversion_rate DESC;
```

### 2. Query to find Duplicate Leads (with comment on schema prevention)

```sql
-- Comment on how the schema prevents duplicates:
-- In the proposed schema.sql, 'lead_id' is defined as the PRIMARY KEY.
-- This means that each 'lead_id' must be unique across all records in the 'leads' table.
-- Any attempt to insert a new row with an existing 'lead_id' will result in a primary key constraint violation,
-- effectively preventing duplicate leads based on their 'lead_id'.
-- If 'crm_record_hash' were also guaranteed unique for each distinct lead in the source CRM,
-- it could optionally be given a UNIQUE constraint in the schema for an additional layer of duplicate prevention.

SELECT
    lead_id,
    COUNT(*) as count_duplicates
FROM
    leads
GROUP BY
    lead_id
HAVING
    COUNT(*) > 1;
```

## Lead Scoring Baseline - Data Cleaning & Preprocessing Decisions (README)

For building a quick baseline model to score lead likelihood to convert, the following decisions were made regarding data cleaning and feature engineering:

*   **`lead_id`**: Dropped as it is a unique identifier and not a predictive feature.
*   **`created_at`**: Converted to datetime objects. Extracted `hour_of_day` and `day_of_week` as new numerical features, as these time-based aspects might influence conversion. The original `created_at` column is then dropped.
*   **Missing Values (Categorical)**:
    *   `area`: Imputed missing values with the string 'Unknown' to retain information that the area was not specified, treating 'Unknown' as its own category.
*   **Missing Values (Numerical)**:
    *   `budget_pkr_lac`, `bedrooms`, `first_response_minutes`, `agent_experience_years`: Imputed missing values with the **median** of their respective columns. The median is chosen over the mean for its robustness to outliers, which is common in messy CRM data.
*   **`crm_record_hash`**: Dropped, as it appears to be a record hash and not a direct predictive feature.
*   **Categorical Feature Encoding**: `source`, `city`, `area`, `property_type` are one-hot encoded to convert them into a format suitable for machine learning models. This creates new binary columns for each unique category.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score
import numpy as np

# Create a copy to perform cleaning without modifying the original DataFrame
df_processed = df_leads.copy()

# 1. Handle 'created_at': Convert to datetime and extract features
df_processed['created_at'] = pd.to_datetime(df_processed['created_at'])
df_processed['hour_of_day'] = df_processed['created_at'].dt.hour
df_processed['day_of_week'] = df_processed['created_at'].dt.dayofweek # Monday=0, Sunday=6

# 2. Handle Missing Values
# Impute 'area' with 'Unknown'
df_processed['area'] = df_processed['area'].fillna('Unknown')

# Impute numerical columns with their median
numerical_cols_to_impute = ['budget_pkr_lac', 'bedrooms', 'first_response_minutes', 'agent_experience_years']
for col in numerical_cols_to_impute:
    if df_processed[col].isnull().any():
        median_val = df_processed[col].median()
        df_processed[col] = df_processed[col].fillna(median_val)

# 3. Drop unnecessary columns: lead_id, crm_record_hash, and original created_at
df_processed = df_processed.drop(columns=['lead_id', 'crm_record_hash', 'created_at'])

# 4. One-hot encode categorical features
categorical_cols = ['source', 'city', 'area', 'property_type']
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

# Define features (X) and target (y)
X = df_processed.drop('converted', axis=1)
y = df_processed['converted']

# Check class balance
class_balance = y.value_counts(normalize=True)
print(f"Class Balance for 'converted' (0=Not Converted, 1=Converted):\n{class_balance}\n")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

# Initialize and train a Logistic Regression model
# Using solver='liblinear' for smaller datasets and for handling L1/L2 regularization well.
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

print("\nLogistic Regression model trained.")

Class Balance for 'converted' (0=Not Converted, 1=Converted):
converted
0    0.930786
1    0.069214
Name: proportion, dtype: float64

Shape of X_train: (7328, 57)
Shape of X_test: (1832, 57)

Logistic Regression model trained.


## Baseline Model Evaluation

**Metric Choice Justification**: Given the class balance observed (e.g., if one class is significantly smaller than the other, which is typical for 'conversion' datasets), **AUC-ROC (Area Under the Receiver Operating Characteristic curve)** is an appropriate metric. Accuracy can be misleading with imbalanced datasets, as a model that simply predicts the majority class would still achieve high accuracy. AUC-ROC, however, evaluates the model's ability to distinguish between classes across various classification thresholds and is robust to class imbalance. A higher AUC-ROC indicates a better-performing model.


In [7]:
# Predict probabilities on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class (converted=1)

# Calculate AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC Score: {auc_roc:.4f}")

# For additional context, let's also look at a classification report and confusion matrix if needed
# To get a classification report, we need to pick a threshold, usually 0.5 for a baseline
# y_pred = (y_pred_proba > 0.5).astype(int)
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))


AUC-ROC Score: 0.9943


In [8]:
# Predict conversion probabilities for the test set
predicted_probabilities = model.predict_proba(X_test)[:, 1]

# Create a DataFrame to display predictions alongside actual values
predictions_df = pd.DataFrame({
    'Actual_Converted': y_test,
    'Predicted_Probability_of_Conversion': predicted_probabilities
})

# Display the first few predictions
print("Sample of Predicted Probabilities vs. Actual Conversion:")
display(predictions_df.head(10))

Sample of Predicted Probabilities vs. Actual Conversion:


,Actual_Converted,Predicted_Probability_of_Conversion
7883,0,1.174962e-03
4570,0,4.906197e-04
904,0,1.116959e-02
1903,0,1.264603e-07
703,0,7.479584e-04
5181,0,4.054822e-05
6777,0,8.871069e-11
4452,0,5.327583e-03
2252,0,1.621000e-09
941,1,1.141964e-02
